**מטלה בלמידת מכונה – ניתוח טקסט (NLP): סיווג חדשות מזויפות**

**שמות הסטודנטים:**  
זוהר א. (####)  
סטודנט/ית נוסף/ת (####)

בפרויקט זה אנו עוסקים בבעיית סיווג טקסט בינארית, שמטרתה לקבוע האם כתבת חדשות שייכת למחלקה 0 או למחלקה 1.  
ה-dataset שנבחר הוא **Fake News Classification** מ-Kaggle, והוא כולל את הקבצים `train.csv`, `test.csv` ו-`evaluation.csv`, כאשר תהליך האימון והבדיקה המרכזי יתבצע על `train` ו-`test` בהתאם לדרישות המטלה.  
במהלך העבודה נבצע טעינת נתונים, עיבוד מקדים לטקסט, ייצוג הטקסט כמאפיינים מספריים, מימוש עצמאי של האלגוריתם **Multinomial Naive Bayes**, ניסויים עם כמה שיטות Feature Engineering ו-hyperparameters, ולבסוף חיזוי והערכת ביצועים באמצעות מדד **F1**.


**שימוש ב-AI ובמקורות עזר**

להלן דוגמאות לפרומפטים ולעזרים שבהם נעשה שימוש במהלך העבודה:

- "הסבר איך עובד Multinomial Naive Bayes עבור סיווג טקסט, כולל priors, likelihoods ו-Laplace smoothing."  
  **מטרה:** להבין לעומק את האלגוריתם לפני המימוש העצמאי.

- "מה ההבדל בין CountVectorizer לבין TfidfVectorizer, ומתי עדיף להשתמש בכל אחד מהם?"  
  **מטרה:** לבחור שיטות Feature Engineering מתאימות ולהשוות ביניהן.

- "איך מקובל להציג Grid Search עם 5-Fold Cross Validation במחברת של מטלת Machine Learning?"  
  **מטרה:** לוודא שהניסויים והתוצאות יוצגו בצורה ברורה ומסודרת.

נעשה שימוש ב-AI לצורך הבנה, תכנון, ניסוח וסיוע טכני, אך המימוש הותאם לפרויקט, נבדק בפועל, והעבודה מבוססת על הבנה של השלבים והתוצאות.


In [ ]:

# %pip install pandas numpy scikit-learn matplotlib jupyter

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_colwidth", 120)


In [ ]:

train = pd.read_csv("data/train.csv", sep=";", index_col=0)
test = pd.read_csv("data/test.csv", sep=";", index_col=0)
evaluation = pd.read_csv("data/evaluation.csv", sep=";", index_col=0)

print("TRAIN HEAD")
display(train.head())

print("TEST HEAD")
display(test.head())

print("EVALUATION HEAD")
display(evaluation.head())


In [ ]:

print("Shapes")
print("Train:", train.shape)
print("Test:", test.shape)
print("Evaluation:", evaluation.shape)

print("\nColumns")
print("Train:", train.columns.tolist())
print("Test:", test.columns.tolist())
print("Evaluation:", evaluation.columns.tolist())

print("\nLabel distribution - train")
display(train["label"].value_counts().sort_index())

print("Label distribution - test")
display(test["label"].value_counts().sort_index())

print("Label distribution - evaluation")
display(evaluation["label"].value_counts().sort_index())

print("Missing values")
display(pd.DataFrame({
    "train_missing": train.isnull().sum(),
    "test_missing": test.isnull().sum(),
    "evaluation_missing": evaluation.isnull().sum()
}))

label_counts = train["label"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(label_counts.index.astype(str), label_counts.values)
plt.title("Train label distribution")
plt.xlabel("Label")
plt.ylabel("Number of samples")
plt.show()


In [ ]:

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(word for word in text.split() if word not in stop_words)
    return text.strip()

for df in (train, test, evaluation):
    df["content_title_text"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
    df["content_text_only"] = df["text"].fillna("").str.strip()

    df["clean_title_text"] = df["content_title_text"].apply(clean_text)
    df["clean_text_only"] = df["content_text_only"].apply(clean_text)


In [ ]:

def show_preprocessing_examples(df, dataset_name, raw_col, clean_col, n=3):
    examples = df[[raw_col, clean_col, "label"]].head(n).copy()
    examples["length_before"] = examples[raw_col].apply(len)
    examples["length_after"] = examples[clean_col].apply(len)
    print(dataset_name)
    display(examples)

show_preprocessing_examples(train, "TRAIN EXAMPLES", "content_title_text", "clean_title_text", n=3)
show_preprocessing_examples(test, "TEST EXAMPLES", "content_title_text", "clean_title_text", n=3)


In [ ]:

for df in (train, test):
    df["char_count"] = df["clean_title_text"].apply(len)
    df["word_count"] = df["clean_title_text"].apply(lambda x: len(x.split()))

print("Train character count summary")
display(train["char_count"].describe())

print("Train word count summary")
display(train["word_count"].describe())

plt.figure(figsize=(8, 4))
plt.hist(train["word_count"], bins=40)
plt.title("Train word count distribution")
plt.xlabel("Word count")
plt.ylabel("Number of samples")
plt.show()


In [ ]:

def create_vectorizer(config_name):
    if config_name == "count":
        return CountVectorizer(max_features=5000)
    elif config_name == "tfidf_uni":
        return TfidfVectorizer(max_features=5000)
    elif config_name == "tfidf_bi":
        return TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    else:
        raise ValueError(f"Unknown vectorizer config: {config_name}")

vectorizer_names = ["count", "tfidf_uni", "tfidf_bi"]
text_modes = {
    "text_only": "clean_text_only",
    "title_plus_text": "clean_title_text"
}

print("Vectorizers:", vectorizer_names)
print("Text modes:", list(text_modes.keys()))

small_examples = train["clean_title_text"].head(3)

for name in vectorizer_names:
    vec = create_vectorizer(name)
    X_small = vec.fit_transform(small_examples)
    feature_names = vec.get_feature_names_out()[:20]
    print(f"\n{name}")
    print("Shape:", X_small.shape)
    print("First 20 features:", feature_names.tolist())


In [ ]:

class MyMultinomialNaiveBayes(BaseEstimator, ClassifierMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = None
        self.feature_log_prob_ = None

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)

        n_classes = len(self.classes_)
        n_features = X.shape[1]

        self.class_log_prior_ = np.zeros(n_classes)
        self.feature_log_prob_ = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            class_count = X_c.shape[0]

            self.class_log_prior_[idx] = np.log(class_count / X.shape[0])

            feature_count = np.asarray(X_c.sum(axis=0)).ravel()
            smoothed_fc = feature_count + self.alpha
            smoothed_total = smoothed_fc.sum()

            self.feature_log_prob_[idx, :] = np.log(smoothed_fc / smoothed_total)

        return self

    def predict_log_proba(self, X):
        log_probs = X @ self.feature_log_prob_.T
        log_probs = np.asarray(log_probs) + self.class_log_prior_
        return log_probs

    def predict(self, X):
        log_probs = self.predict_log_proba(X)
        class_indices = np.argmax(log_probs, axis=1)
        return self.classes_[class_indices]

    def predict_proba(self, X):
        log_probs = self.predict_log_proba(X)
        max_log = np.max(log_probs, axis=1, keepdims=True)
        probs = np.exp(log_probs - max_log)
        probs /= probs.sum(axis=1, keepdims=True)
        return probs


In [ ]:

baseline_vectorizer = create_vectorizer("tfidf_uni")

X_train_baseline = baseline_vectorizer.fit_transform(train["clean_title_text"])
X_test_baseline = baseline_vectorizer.transform(test["clean_title_text"])

y_train = train["label"].values
y_test = test["label"].values

baseline_model = MyMultinomialNaiveBayes(alpha=1.0)
baseline_model.fit(X_train_baseline, y_train)

baseline_predictions = baseline_model.predict(X_test_baseline)
baseline_probabilities = baseline_model.predict_proba(X_test_baseline)

baseline_f1 = f1_score(y_test, baseline_predictions, average="binary")

print("Baseline F1:", round(baseline_f1, 4))
print("First 5 predictions:")
print(baseline_predictions[:5])


In [ ]:

print("Baseline classification report")
print(classification_report(y_test, baseline_predictions))

print("Baseline confusion matrix")
cm_baseline = confusion_matrix(y_test, baseline_predictions)
display(pd.DataFrame(cm_baseline,
                     index=["Actual 0", "Actual 1"],
                     columns=["Pred 0", "Pred 1"]))

flow_examples = test[["title", "text", "clean_title_text", "label"]].head(3).copy()
flow_examples["prediction"] = baseline_predictions[:3]
flow_examples["prob_class_0"] = baseline_probabilities[:3, 0]
flow_examples["prob_class_1"] = baseline_probabilities[:3, 1]

print("Flow examples")
display(flow_examples)


In [ ]:

vectorizer_configs = ["count", "tfidf_uni", "tfidf_bi"]
alpha_values = [0.5, 1.0, 1.5]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y = train["label"].reset_index(drop=True).values
results = []

for text_mode_name, text_col in text_modes.items():
    X_text = train[text_col].reset_index(drop=True)

    for vec_name in vectorizer_configs:
        for alpha in alpha_values:
            fold_scores = []

            for train_idx, valid_idx in cv.split(X_text, y):
                X_train_fold_text = X_text.iloc[train_idx]
                X_valid_fold_text = X_text.iloc[valid_idx]
                y_train_fold = y[train_idx]
                y_valid_fold = y[valid_idx]

                vectorizer = create_vectorizer(vec_name)
                X_train_fold = vectorizer.fit_transform(X_train_fold_text)
                X_valid_fold = vectorizer.transform(X_valid_fold_text)

                model = MyMultinomialNaiveBayes(alpha=alpha)
                model.fit(X_train_fold, y_train_fold)
                preds = model.predict(X_valid_fold)

                score = f1_score(y_valid_fold, preds, average="binary")
                fold_scores.append(score)

            results.append({
                "text_mode": text_mode_name,
                "vectorizer": vec_name,
                "alpha": alpha,
                "mean_f1_cv": np.mean(fold_scores),
                "std_f1_cv": np.std(fold_scores)
            })

results_df = pd.DataFrame(results).sort_values(by="mean_f1_cv", ascending=False).reset_index(drop=True)
display(results_df)


In [ ]:

best_result = results_df.iloc[0]
best_text_mode_name = best_result["text_mode"]
best_text_col = text_modes[best_text_mode_name]
best_vectorizer_name = best_result["vectorizer"]
best_alpha = float(best_result["alpha"])

print("Best configuration")
display(best_result)

best_vectorizer = create_vectorizer(best_vectorizer_name)

X_train_best = best_vectorizer.fit_transform(train[best_text_col])
X_test_best = best_vectorizer.transform(test[best_text_col])

best_model = MyMultinomialNaiveBayes(alpha=best_alpha)
best_model.fit(X_train_best, y_train)

test_predictions = best_model.predict(X_test_best)
test_probabilities = best_model.predict_proba(X_test_best)

final_f1 = f1_score(y_test, test_predictions, average="binary")

print("Final test F1:", round(final_f1, 4))
print("First 5 test predictions:")
print(test_predictions[:5])


In [ ]:

comparison_df = test[["title", "label"]].copy()
comparison_df["prediction"] = test_predictions
comparison_df["prob_class_0"] = test_probabilities[:, 0]
comparison_df["prob_class_1"] = test_probabilities[:, 1]

print("First 5 test samples with predictions")
display(comparison_df.head(5))

print("Final classification report")
print(classification_report(y_test, test_predictions))

print("Final confusion matrix")
cm = confusion_matrix(y_test, test_predictions)
display(pd.DataFrame(cm,
                     index=["Actual 0", "Actual 1"],
                     columns=["Pred 0", "Pred 1"]))


In [ ]:

top_results = results_df.head(10).copy()
print("Top 10 configurations")
display(top_results)

best_by_text_mode = results_df.groupby("text_mode", as_index=False)["mean_f1_cv"].max()
print("Best score by text mode")
display(best_by_text_mode)


In [ ]:

feature_names = best_vectorizer.get_feature_names_out()

log_prob_df = pd.DataFrame({
    "feature": feature_names,
    "log_prob_class_0": best_model.feature_log_prob_[0],
    "log_prob_class_1": best_model.feature_log_prob_[1]
})

log_prob_df["difference_1_minus_0"] = log_prob_df["log_prob_class_1"] - log_prob_df["log_prob_class_0"]
log_prob_df["difference_0_minus_1"] = log_prob_df["log_prob_class_0"] - log_prob_df["log_prob_class_1"]

top_class_1 = log_prob_df.sort_values("difference_1_minus_0", ascending=False).head(15)
top_class_0 = log_prob_df.sort_values("difference_0_minus_1", ascending=False).head(15)

print("Top features for class 1")
display(top_class_1[["feature", "difference_1_minus_0"]])

print("Top features for class 0")
display(top_class_0[["feature", "difference_0_minus_1"]])


In [ ]:

print("Note:")
print("evaluation.csv was loaded for inspection only.")
print("The main training and testing flow in this project is based on train.csv and test.csv.")
